# Import Library

In [1]:
import pandas as pd

# Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load the dataset

In [3]:
file_path = '/content/drive/MyDrive/FYP Data Cleaning/Raw Data/personal-finance-budgeting-records.csv'
df2 = pd.read_csv(file_path)

# Display initial shape and info of the dataset

In [4]:
print("--- Initial Shape ---")
print(df2.shape)

print("\n--- Info & Data Types ---")
df2.info()

--- Initial Shape ---
(200, 18)

--- Info & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   household_id          200 non-null    object 
 1   record_id             200 non-null    object 
 2   record_type           200 non-null    object 
 3   category              200 non-null    object 
 4   subcategory           132 non-null    object 
 5   amount                200 non-null    float64
 6   currency              200 non-null    object 
 7   transaction_date      200 non-null    object 
 8   description           200 non-null    object 
 9   payment_method        200 non-null    object 
 10  payer_payee           200 non-null    object 
 11  is_recurring          200 non-null    bool   
 12  recurrence_interval   86 non-null     object 
 13  household_size        200 non-null    int64  
 14  location_city  

# Check for missing values in each column

In [5]:
print("\n--- Missing Values ---")
print(df2.isnull().sum())


--- Missing Values ---
household_id              0
record_id                 0
record_type               0
category                  0
subcategory              68
amount                    0
currency                  0
transaction_date          0
description               0
payment_method            0
payer_payee               0
is_recurring              0
recurrence_interval     114
household_size            0
location_city             0
location_state            0
location_postal_code      0
location_country          0
dtype: int64


# Check for Duplicates count

In [6]:
dup_rows      = df2.duplicated().sum()
dup_record_id = df2.duplicated(subset=['record_id']).sum()

print("--- Duplicate Check (before cleaning) ---")
print(f"Total rows                       : {len(df2)}")
print(f"Duplicate rows (all columns)     : {dup_rows}")
print(f"Duplicate rows (by record_id)    : {dup_record_id}")

--- Duplicate Check (before cleaning) ---
Total rows                       : 200
Duplicate rows (all columns)     : 0
Duplicate rows (by record_id)    : 50


# Handle missing values

Fill missing subcategories with 'General'

In [7]:
df2['subcategory'] = df2['subcategory'].fillna('General')
df2['recurrence_interval'] = df2['recurrence_interval'].fillna('One-time')

# Remove duplicate records

In [8]:
before = len(df2)
df2 = df2.drop_duplicates().reset_index(drop=True)
print(f"Removed   : {before - len(df2)}")
print(f"Remaining : {len(df2)}")

Removed   : 0
Remaining : 200


# Standardize data types

In [9]:
df2['transaction_date'] = pd.to_datetime(df2['transaction_date'], errors='coerce')
print("Invalid dates (NaT):", df2['transaction_date'].isna().sum())

Invalid dates (NaT): 0


# Check the currency

In [10]:
df2['currency'] = df2['currency'].str.strip().str.upper()
print(df2['currency'].value_counts(dropna=False))

currency
USD    136
EUR     21
GBP     10
JPY      9
INR      9
AUD      8
CAD      7
Name: count, dtype: int64


# Convert into LKR

In [13]:
RATE_DATE   = "2026-10-14"
RATE_SOURCE = "Central Bank of Sri Lanka (daily indicative rates)"

RATES_TO_LKR = {
    'USD': 331.0482,
    'EUR': 370.9726,
    'GBP': 439.2016,
    'JPY': 2.0902,
    'INR': 3.4744,
    'AUD': 231.5020,
    'CAD': 233.0341,
    'LKR': 1.0,
}

# Make sure every currency used in the dataset has a rate
used = set(df2['currency'].unique())
missing_rates = [c for c in used if RATES_TO_LKR.get(c) is None]
if missing_rates:
    raise ValueError(f"No rate available for these currencies: {missing_rates}. Fill them in RATES_TO_LKR.")

# Add a new LKR column; the original 'amount' and 'currency' columns stay unchanged
df2['amount_LKR'] = (df2['amount'] * df2['currency'].map(RATES_TO_LKR)).round(2)

print(f"Rates: {RATE_SOURCE}, {RATE_DATE}")
print(df2[['amount', 'currency', 'amount_LKR']].head(10))
print("Missing amount_LKR:", df2['amount_LKR'].isna().sum())

Rates: Central Bank of Sri Lanka (daily indicative rates), 2026-10-14
    amount currency  amount_LKR
0  4200.00      USD  1390402.44
1  1850.00      USD   612439.17
2   350.25      USD   115949.63
3    94.13      USD    31161.57
4   120.55      USD    39907.86
5   420.00      USD   139040.24
6    48.50      USD    16055.84
7    62.70      USD    20756.72
8  2300.00      USD   761410.86
9   110.00      USD    36415.30
Missing amount_LKR: 0


# Verify final shape and view sample data

In [16]:
print("--- Cleaned Shape ---")
print(df2.shape)

--- Cleaned Shape ---
(200, 19)


In [17]:
print("\n--- Remaining Missing Values ---")
print(df2.isnull().sum())


--- Remaining Missing Values ---
household_id            0
record_id               0
record_type             0
category                0
subcategory             0
amount                  0
currency                0
transaction_date        0
description             0
payment_method          0
payer_payee             0
is_recurring            0
recurrence_interval     0
household_size          0
location_city           0
location_state          0
location_postal_code    0
location_country        0
amount_LKR              0
dtype: int64


In [18]:
# Manual check: row 0 = 4200 USD x 330.89 = 1,389,738.00
print("\nRow 0 check:", df2.loc[0, 'amount'], df2.loc[0, 'currency'], "->", df2.loc[0, 'amount_LKR'])

print("\n--- Sample ---")
print(df2.head())


Row 0 check: 4200.0 USD -> 1390402.44

--- Sample ---
  household_id record_id record_type    category  subcategory   amount  \
0        HH001    REC001      income      salary      General  4200.00   
1        HH002    REC002     expense        rent      General  1850.00   
2        HH003    REC003      income  investment    dividends   350.25   
3        HH004    REC004     expense   groceries       fruits    94.13   
4        HH005    REC005     expense   utilities  electricity   120.55   

  currency transaction_date                     description payment_method  \
0      USD       2023-11-30  November salary direct deposit  bank_transfer   
1      USD       2023-12-01         December apartment rent  bank_transfer   
2      USD       2024-01-15       Quarterly stock dividends  bank_transfer   
3      USD       2023-12-14       Grocery shopping - fruits     debit_card   
4      USD       2024-01-05        Electricity bill January  bank_transfer   

            payer_payee  is_rec

# Save the cleaned dataset

In [19]:
output_path = '/content/drive/MyDrive/FYP Data Cleaning/Clean Data/personal_finance_budgeting_cleaned.csv'
df2.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /content/drive/MyDrive/FYP Data Cleaning/Clean Data/personal_finance_budgeting_cleaned.csv
